# Refuerzo: Fundamentos de Clustering
## Ingeniería del Conocimiento (ISO56B) — UNCP
### Msc. Jaime Antonio Huaytalla Pariona — Semestre 2026-II

**Objetivo:** Comprender los conceptos fundamentales de clustering (aprendizaje no supervisado), desde la idea básica de "encontrar grupos" hasta la aplicación de K-Means y DBSCAN.

**Nivel:** Básico → Intermedio

**Contenido:**
1. ¿Qué es clustering? Diferencia con clasificación
2. Datos sintéticos para entender visualmente
3. K-Means paso a paso
4. Selección de K: Codo y Silueta
5. DBSCAN: clustering basado en densidad
6. Clustering Jerárquico y Dendrogramas
7. Métricas de evaluación
8. Visualización con PCA y t-SNE


## 1. Configuración del Entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
np.random.seed(42)

print("✅ Librerías cargadas correctamente")


## 2. ¿Qué es Clustering?

El clustering busca **agrupar datos similares** sin etiquetas previas. Es **aprendizaje NO supervisado**: nadie nos dice la respuesta correcta.

Vamos a crear datos sintéticos para entender visualmente cómo funcionan los algoritmos.


In [ ]:
from sklearn.datasets import make_blobs, make_moons, make_circles

# Crear 3 tipos de datos con formas diferentes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# a) Blobs (esferas) — ideal para K-Means
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', s=30, alpha=0.7)
axes[0].set_title('Blobs (3 grupos esféricos)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# b) Lunas — NO ideal para K-Means
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', s=30, alpha=0.7)
axes[1].set_title('Lunas (2 medias lunas)')

# c) Círculos — NO ideal para K-Means
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
axes[2].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', s=30, alpha=0.7)
axes[2].set_title('Círculos concéntricos')

plt.suptitle('Tres formas de datos — ¿Qué algoritmo funciona mejor para cada una?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Pregunta clave: ¿Un mismo algoritmo funcionará igual de bien con las 3 formas?")
print("Spoiler: NO. K-Means funciona bien con blobs, pero no con lunas ni círculos.")


## 3. K-Means Paso a Paso

K-Means agrupa los datos alrededor de K **centroides** (centros).

### El algoritmo:
1. Elegir K centroides iniciales
2. Asignar cada punto al centroide más cercano
3. Recalcular los centroides como la media de sus puntos
4. Repetir 2-3 hasta que no cambie nada

Vamos a visualizarlo iteración por iteración.


In [ ]:
from sklearn.cluster import KMeans

# Visualizar K-Means iteración por iteración con los blobs
X = X_blobs.copy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, n_iter in enumerate([1, 2, 3, 5, 10, 50]):
    ax = axes.ravel()[i]
    km = KMeans(n_clusters=3, init='random', n_init=1, max_iter=n_iter, random_state=0)
    km.fit(X)
    labels = km.labels_
    centers = km.cluster_centers_

    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=20, alpha=0.5)
    ax.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200,
               edgecolors='black', linewidths=2, label='Centroides')
    ax.set_title(f'Iteración {n_iter} | Inercia={km.inertia_:.1f}')
    ax.legend(fontsize=8)

plt.suptitle('K-Means: Evolución de los Centroides', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observar cómo los centroides (X rojas) se mueven hasta estabilizarse.")
print("La inercia (suma de distancias al cuadrado) decrece con cada iteración.")


### 3.1 Aplicar K-Means con scikit-learn


In [ ]:
# K-Means con la configuración recomendada
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
kmeans.fit(X_blobs)

print("Resultados de K-Means:")
print(f"  Clusters: {len(set(kmeans.labels_))}")
print(f"  Muestras por cluster: {np.bincount(kmeans.labels_)}")
print(f"  Inercia (WCSS): {kmeans.inertia_:.2f}")
print(f"  Iteraciones hasta convergencia: {kmeans.n_iter_}")

# Visualizar resultado
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans.labels_, cmap='viridis', s=30, alpha=0.6)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='red', marker='X', s=200, edgecolors='black', linewidths=2, label='Centroides')
ax.set_title('K-Means: Resultado Final (K=3)')
ax.legend()
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()


## 4. ¿Cómo Elegir K? — Método del Codo y Silueta


In [ ]:
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del Codo
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='K=3 (codo)')
axes[0].set_xlabel('K (número de clusters)')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_title('Método del Codo')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Coeficiente de Silueta
axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', label='K=3 (máximo)')
axes[1].set_xlabel('K (número de clusters)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Coeficiente de Silueta')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Selección del Número Óptimo de Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"Según el codo: K=3 (donde la curva deja de bajar rápido)")
print(f"Según silueta: K={best_k} (mayor score = {max(silhouettes):.3f})")


## 5. DBSCAN — Clustering Basado en Densidad

DBSCAN no necesita especificar K. Encuentra clusters de cualquier forma detectando regiones densas.

Veamos cómo funciona con datos de "lunas" donde K-Means falla.


In [ ]:
from sklearn.cluster import DBSCAN

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means en lunas (FALLA)
km = KMeans(n_clusters=2, random_state=42)
labels_km = km.fit_predict(X_moons)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km, cmap='viridis', s=30)
axes[0].set_title(f'K-Means en Lunas\n(Sil: {silhouette_score(X_moons, labels_km):.3f}) — FALLA')

# DBSCAN en lunas (FUNCIONA)
db = DBSCAN(eps=0.2, min_samples=5)
labels_db = db.fit_predict(X_moons)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db, cmap='viridis', s=30)
axes[1].set_title(f'DBSCAN en Lunas\n({n_clusters} clusters, {sum(labels_db==-1)} ruido) — FUNCIONA')

# DBSCAN en círculos
db2 = DBSCAN(eps=0.2, min_samples=5)
labels_db2 = db2.fit_predict(X_circles)
n_clusters2 = len(set(labels_db2)) - (1 if -1 in labels_db2 else 0)
axes[2].scatter(X_circles[:, 0], X_circles[:, 1], c=labels_db2, cmap='viridis', s=30)
axes[2].set_title(f'DBSCAN en Círculos\n({n_clusters2} clusters) — FUNCIONA')

plt.suptitle('K-Means vs DBSCAN: Formas No Esféricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Conclusión clave:")
print("• K-Means asume clusters esféricos → falla con lunas y círculos")
print("• DBSCAN encuentra clusters de CUALQUIER forma")
print("• DBSCAN también detecta puntos de RUIDO (label = -1)")


In [ ]:
# Efecto de eps en DBSCAN
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
eps_values = [0.05, 0.15, 0.3, 0.8]

for ax, eps in zip(axes, eps_values):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = sum(labels == -1)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', s=20)
    ax.set_title(f'eps={eps}\n{n_clusters} clusters, {n_noise} ruido')

plt.suptitle('DBSCAN: Efecto del parámetro eps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("• eps muy pequeño → muchos clusters pequeños + mucho ruido")
print("• eps muy grande → todo en un solo cluster")
print("• Buscar el valor intermedio que capture la estructura real")


## 6. Clustering Jerárquico y Dendrogramas

El clustering jerárquico construye un árbol (dendrograma) que muestra cómo los clusters se fusionan paso a paso.


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

# Usar un subconjunto para el dendrograma (más legible)
X_sub = X_blobs[:50]

Z = linkage(X_sub, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Dendrograma
dendrogram(Z, ax=axes[0], leaf_rotation=90, leaf_font_size=8)
axes[0].set_title('Dendrograma (method=ward)')
axes[0].set_xlabel('Muestras')
axes[0].set_ylabel('Distancia')
axes[0].axhline(y=8, color='red', linestyle='--', label='Corte para K=3')
axes[0].legend()

# Resultado con K=3
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_blobs)
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_agg, cmap='viridis', s=30, alpha=0.6)
axes[1].set_title('Clustering Jerárquico Aglomerativo (K=3)')

plt.suptitle('Clustering Jerárquico', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("El dendrograma muestra la jerarquía completa de fusiones.")
print("Cortando a cierta altura, obtenemos K clusters.")


## 7. Métricas de Evaluación para Clustering

¿Cómo saber si nuestros clusters son "buenos" sin etiquetas?


In [ ]:
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                           davies_bouldin_score)

# Comparar métricas para diferentes algoritmos en blobs
algoritmos = {
    'K-Means (K=3)': KMeans(n_clusters=3, random_state=42).fit_predict(X_blobs),
    'K-Means (K=5)': KMeans(n_clusters=5, random_state=42).fit_predict(X_blobs),
    'DBSCAN (eps=1.5)': DBSCAN(eps=1.5, min_samples=5).fit_predict(X_blobs),
    'Jerárquico (K=3)': AgglomerativeClustering(n_clusters=3).fit_predict(X_blobs),
}

print(f"{'Algoritmo':<22s} {'Silhouette':>12s} {'Calinski-H':>12s} {'Davies-B':>12s}")
print("=" * 62)

for nombre, labels in algoritmos.items():
    # Excluir ruido para métricas (DBSCAN)
    mask = labels != -1
    if sum(mask) < len(labels) and len(set(labels[mask])) > 1:
        sil = silhouette_score(X_blobs[mask], labels[mask])
        ch  = calinski_harabasz_score(X_blobs[mask], labels[mask])
        db  = davies_bouldin_score(X_blobs[mask], labels[mask])
    elif len(set(labels)) > 1:
        sil = silhouette_score(X_blobs, labels)
        ch  = calinski_harabasz_score(X_blobs, labels)
        db  = davies_bouldin_score(X_blobs, labels)
    else:
        sil, ch, db = 0, 0, 999
    print(f"{nombre:<22s} {sil:12.3f} {ch:12.1f} {db:12.3f}")

print("\nInterpretación:")
print("• Silhouette: más alto = mejor (máx 1)")
print("• Calinski-Harabasz: más alto = mejor")
print("• Davies-Bouldin: más bajo = mejor (mín 0)")


## 8. Visualización de Clusters con PCA y t-SNE

Con datos de alta dimensión, usamos reducción de dimensionalidad para visualizar en 2D.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_iris

# Usar Iris (4 dimensiones) para demostrar
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

# Clustering
km_iris = KMeans(n_clusters=3, random_state=42)
labels_iris = km_iris.fit_predict(X_iris)

# Reducir a 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_iris)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original (2 features)
axes[0].scatter(X_iris[:, 0], X_iris[:, 1], c=labels_iris, cmap='viridis', s=40, alpha=0.7)
axes[0].set_title('2 primeros features originales')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# PCA
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_iris, cmap='viridis', s=40, alpha=0.7)
axes[1].set_title(f'PCA (varianza explicada: {pca.explained_variance_ratio_.sum():.1%})')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

# t-SNE
axes[2].scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_iris, cmap='viridis', s=40, alpha=0.7)
axes[2].set_title('t-SNE')
axes[2].set_xlabel('Dim 1')
axes[2].set_ylabel('Dim 2')

plt.suptitle('Visualización de Clusters (Iris, 4D → 2D)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Comparar con etiquetas reales
print("Comparación clusters K-Means vs clases reales de Iris:")
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
print(f"  Adjusted Rand Index: {adjusted_rand_score(iris.target, labels_iris):.3f}")
print(f"  Normalized Mutual Info: {normalized_mutual_info_score(iris.target, labels_iris):.3f}")
print("\nValores cercanos a 1 indican que los clusters coinciden bien con las clases reales.")


## 9. Resumen

| Algoritmo | Tipo | K necesario | Formas | Ruido |
|-----------|------|-------------|--------|-------|
| **K-Means** | Centroides | Sí | Esféricas | No detecta |
| **DBSCAN** | Densidad | No | Cualquiera | Sí detecta |
| **Jerárquico** | Aglomerativo | Sí (o ver dendrograma) | Depende del linkage | No detecta |

### Próximo notebook:
- Dataset real (Mall Customers / Wine)
- Comparación completa de algoritmos
- Despliegue con Flask y Gradio

---
*Material de refuerzo — Ingeniería del Conocimiento (ISO56B) — UNCP*
